# 13 — Curvature

**Engineering question**

Do sentence trajectories become straighter from early to middle transformer layers?

Notebook 07 extracted hidden-state trajectories:

\[
x_1^\ell, x_2^\ell, \ldots, x_n^\ell
\]

This notebook computes the paper's trajectory geometry.

For adjacent displacement vectors,

\[
v_i^\ell = x_{i+1}^\ell - x_i^\ell
\]

the turning angle is:

\[
\theta_i^\ell =
\arccos
\left(
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\right)
\]

Average sentence curvature is the mean turning angle across the sentence.

Lower average angle means a straighter trajectory.


## Setup

This notebook can run from the repo root or inside `notebooks/`.

It tries to load the saved hidden states from Notebook 07:

```text
results/arrays/07_hidden_states.npz
```

If that file is missing, the notebook runs a compact inference pass using `distilgpt2`.

Outputs:

```text
figures/
results/csv/
results/json/
results/13_outputs.zip
```


In [ ]:
from pathlib import Path
import json
import zipfile
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"
ARR = RES / "arrays"

for p in [FIG, RES, CSV, JS, ARR]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Load hidden states

Use the Notebook 07 output if available.

Otherwise, run a small inference pass and save the same array format.


In [ ]:
npz_path = ARR / "07_hidden_states.npz"

if npz_path.exists():
    data = np.load(npz_path, allow_pickle=True)
    hidden = data["hidden"]
    input_ids = data["input_ids"]
    attention_mask = data["attention_mask"]
    sentences = list(data["sentences"])
    print("loaded:", npz_path)
else:
    print("No Notebook 07 hidden-state file found. Running compact inference.")

    MODEL_NAME = "distilgpt2"
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_hidden_states=True)
    model.to(DEVICE)
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    sentences = [
        "Trajectory straightening specifies language prediction.",
        "The cat sat on the mat.",
        "Large language models predict the next token.",
        "Geometry organizes predictive representations.",
        "Curvature decreases across transformer depth.",
    ]

    batch = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(**batch, output_hidden_states=True)

    hidden = torch.stack(outputs.hidden_states, dim=0).detach().cpu().numpy()
    input_ids = batch["input_ids"].detach().cpu().numpy()
    attention_mask = batch["attention_mask"].detach().cpu().numpy()

    np.savez_compressed(
        npz_path,
        hidden=hidden,
        input_ids=input_ids,
        attention_mask=attention_mask,
        sentences=np.array(sentences, dtype=object),
    )
    print("saved:", npz_path)

print("hidden shape:", hidden.shape)
print("sentences:", len(sentences))


## 2. Curvature helpers

The paper's curvature metric is an average turning angle along the sentence trajectory.

This notebook computes:

- adjacent displacement vectors,
- adjacent-step cosine,
- turning angle in radians,
- turning angle in degrees,
- sentence-average curvature by layer.


In [ ]:
def active_hidden_for_sentence(sentence_index, layer_index):
    mask = attention_mask[sentence_index].astype(bool)
    return hidden[layer_index, sentence_index, mask, :]

def step_vectors(points):
    return np.diff(points, axis=0)

def adjacent_step_cosines(points, eps=1e-12):
    v = step_vectors(points)
    if len(v) < 2:
        return np.array([])
    a = v[:-1]
    b = v[1:]
    denom = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    denom = np.maximum(denom, eps)
    cos = np.sum(a * b, axis=1) / denom
    return np.clip(cos, -1.0, 1.0)

def turning_angles(points):
    cos = adjacent_step_cosines(points)
    return np.arccos(cos)

def trajectory_length(points):
    v = step_vectors(points)
    return np.linalg.norm(v, axis=1).sum()

def mean_curvature_angle(points):
    theta = turning_angles(points)
    if len(theta) == 0:
        return np.nan
    return theta.mean()


## 3. Compute layerwise curvature

For each sentence and layer, compute the average turning angle.

This gives the central quantity:

\[
C^\ell_s =
\frac{1}{k}
\sum_i
\theta_i^\ell
\]


In [ ]:
records = []
angle_records = []

for s_idx, sentence in enumerate(sentences):
    for layer in range(hidden.shape[0]):
        points = active_hidden_for_sentence(s_idx, layer)
        theta = turning_angles(points)
        cos = adjacent_step_cosines(points)

        records.append(
            {
                "sentence_index": s_idx,
                "sentence": sentence,
                "layer": layer,
                "active_tokens": int(points.shape[0]),
                "trajectory_length": float(trajectory_length(points)),
                "mean_cosine": float(cos.mean()) if len(cos) else np.nan,
                "mean_curvature_rad": float(theta.mean()) if len(theta) else np.nan,
                "mean_curvature_deg": float(np.degrees(theta.mean())) if len(theta) else np.nan,
                "median_curvature_deg": float(np.degrees(np.median(theta))) if len(theta) else np.nan,
            }
        )

        for local_i, (c, th) in enumerate(zip(cos, theta)):
            angle_records.append(
                {
                    "sentence_index": s_idx,
                    "sentence": sentence,
                    "layer": layer,
                    "turn_index": local_i,
                    "adjacent_step_cosine": float(c),
                    "turning_angle_rad": float(th),
                    "turning_angle_deg": float(np.degrees(th)),
                }
            )

curvature = pd.DataFrame(records)
angles = pd.DataFrame(angle_records)

curvature.to_csv(CSV / "13_curvature_by_sentence_layer.csv", index=False)
angles.to_csv(CSV / "13_turning_angles.csv", index=False)

curvature.to_json(JS / "13_curvature_by_sentence_layer.json", orient="records", indent=2)
angles.to_json(JS / "13_turning_angles.json", orient="records", indent=2)

curvature.head()


## 4. Curvature profile by layer

This is the first paper-style profile.

A decrease from early to middle layers corresponds to trajectory straightening.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for s_idx in sorted(curvature["sentence_index"].unique()):
    subset = curvature[curvature["sentence_index"] == s_idx]
    ax.plot(
        subset["layer"],
        subset["mean_curvature_deg"],
        marker="o",
        markersize=3.5,
        linewidth=1.5,
        alpha=0.7,
        label=f"sentence {s_idx}",
    )

mean_profile = curvature.groupby("layer")["mean_curvature_deg"].mean().reset_index()
ax.plot(
    mean_profile["layer"],
    mean_profile["mean_curvature_deg"],
    marker="o",
    linewidth=3.0,
    color="black",
    label="batch mean",
)

ax.set_title("Mean Sentence Curvature Across Layers")
ax.set_xlabel("layer")
ax.set_ylabel("mean turning angle (degrees)")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)

savefig(fig, "13_curvature_by_layer.png")
plt.show()


## 5. Curvature change relative to the first layer

The paper often reports curvature change relative to the first layer.

For each sentence:

\[
\Delta C^\ell_s = C^\ell_s - C^0_s
\]

Negative values indicate lower curvature than the first layer.


In [ ]:
curvature = curvature.copy()
baseline = curvature[curvature["layer"] == 0][["sentence_index", "mean_curvature_deg"]]
baseline = baseline.rename(columns={"mean_curvature_deg": "baseline_curvature_deg"})

curvature = curvature.merge(baseline, on="sentence_index", how="left")
curvature["curvature_change_deg"] = (
    curvature["mean_curvature_deg"] - curvature["baseline_curvature_deg"]
)

curvature.to_csv(CSV / "13_curvature_change_by_sentence_layer.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))

for s_idx in sorted(curvature["sentence_index"].unique()):
    subset = curvature[curvature["sentence_index"] == s_idx]
    ax.plot(
        subset["layer"],
        subset["curvature_change_deg"],
        marker="o",
        markersize=3.5,
        linewidth=1.5,
        alpha=0.65,
        label=f"sentence {s_idx}",
    )

mean_change = curvature.groupby("layer")["curvature_change_deg"].mean().reset_index()
ax.plot(
    mean_change["layer"],
    mean_change["curvature_change_deg"],
    marker="o",
    linewidth=3.0,
    color="black",
    label="batch mean",
)

ax.axhline(0, linewidth=1)
ax.set_title("Curvature Change Relative to First Layer")
ax.set_xlabel("layer")
ax.set_ylabel("curvature change (degrees)")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)

savefig(fig, "13_curvature_change_by_layer.png")
plt.show()


## 6. Sentence × layer heatmap

A heatmap makes it easier to see whether curvature reduction is consistent across sentences.


In [ ]:
heat = curvature.pivot(index="sentence_index", columns="layer", values="curvature_change_deg")

fig, ax = plt.subplots(figsize=(9, 3.8))
im = ax.imshow(heat.values, aspect="auto")

ax.set_title("Curvature Change Heatmap")
ax.set_xlabel("layer")
ax.set_ylabel("sentence index")
ax.set_xticks(np.arange(heat.shape[1]))
ax.set_xticklabels(heat.columns)
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_yticklabels(heat.index)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("curvature change (degrees)")

savefig(fig, "13_curvature_heatmap.png")
plt.show()


## 7. Turning-angle distribution

Compare early, middle, and final layers.

If straightening is present, the middle-layer distribution should shift toward lower turning angles.


In [ ]:
max_layer = int(curvature["layer"].max())
early_layer = 0
middle_layer = int(curvature.groupby("layer")["mean_curvature_deg"].mean().idxmin())
final_layer = max_layer

selected = {
    "early": early_layer,
    "middle/min": middle_layer,
    "final": final_layer,
}

fig, ax = plt.subplots(figsize=(8.5, 4.8))

bins = np.linspace(0, 180, 30)

for label, layer in selected.items():
    vals = angles[angles["layer"] == layer]["turning_angle_deg"].dropna().values
    ax.hist(vals, bins=bins, alpha=0.45, label=f"{label} layer {layer}")

ax.set_title("Turning-Angle Distribution by Layer Region")
ax.set_xlabel("turning angle (degrees)")
ax.set_ylabel("count")
ax.grid(True, alpha=0.25)
ax.legend()

savefig(fig, "13_turning_angle_distribution.png")
plt.show()

selected


## 8. Alignment versus curvature

Adjacent-step cosine and turning angle are equivalent through:

\[
\theta = \arccos(\cos)
\]

This plot verifies that the Notebook 07 alignment quantity correctly maps onto curvature.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.2))

x = angles["adjacent_step_cosine"].values
y = angles["turning_angle_deg"].values

ax.scatter(x, y, alpha=0.45)
ax.set_title("Adjacent-Step Cosine Maps to Turning Angle")
ax.set_xlabel("adjacent-step cosine")
ax.set_ylabel("turning angle (degrees)")
ax.grid(True, alpha=0.25)

savefig(fig, "13_alignment_vs_curvature.png")
plt.show()


## 9. Minimum-curvature layer

Identify the layer with the lowest mean curvature for each sentence and for the batch mean.

This gives a compact summary of where the model's trajectory is straightest.


In [ ]:
min_by_sentence = (
    curvature.loc[curvature.groupby("sentence_index")["mean_curvature_deg"].idxmin()]
    [["sentence_index", "sentence", "layer", "mean_curvature_deg", "curvature_change_deg"]]
    .rename(columns={"layer": "min_curvature_layer"})
    .reset_index(drop=True)
)

batch_mean = curvature.groupby("layer").agg(
    mean_curvature_deg=("mean_curvature_deg", "mean"),
    mean_curvature_change_deg=("curvature_change_deg", "mean"),
).reset_index()

batch_min_layer = int(batch_mean.loc[batch_mean["mean_curvature_deg"].idxmin(), "layer"])

summary = {
    "batch_min_curvature_layer": batch_min_layer,
    "batch_min_curvature_deg": float(batch_mean["mean_curvature_deg"].min()),
    "num_sentences": int(curvature["sentence_index"].nunique()),
    "num_layers": int(curvature["layer"].nunique()),
}

min_by_sentence.to_csv(CSV / "13_min_curvature_layer_by_sentence.csv", index=False)
batch_mean.to_csv(CSV / "13_batch_mean_curvature.csv", index=False)

with open(JS / "13_curvature_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary, min_by_sentence


## 10. Compact interpretation

This notebook computes the paper's core geometric object:

\[
C_s^\ell =
\frac{1}{k}
\sum_i
\arccos
\left(
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\right)
\]

For a small `distilgpt2` sentence batch, the output should be interpreted as a pipeline validation, not a full reproduction.

The important result is that the repo can now compute:

```text
hidden states
→ displacement vectors
→ adjacent-step cosines
→ turning angles
→ layerwise sentence curvature
```

The next notebook can compare trained and untrained models or scale the sentence batch.


## 11. Export and download

Package Notebook 13 outputs.


In [ ]:
zip_path = RES / "13_outputs.zip"

files_to_zip = [
    FIG / "13_curvature_by_layer.png",
    FIG / "13_curvature_change_by_layer.png",
    FIG / "13_curvature_heatmap.png",
    FIG / "13_turning_angle_distribution.png",
    FIG / "13_alignment_vs_curvature.png",
    CSV / "13_curvature_by_sentence_layer.csv",
    CSV / "13_turning_angles.csv",
    CSV / "13_curvature_change_by_sentence_layer.csv",
    CSV / "13_min_curvature_layer_by_sentence.csv",
    CSV / "13_batch_mean_curvature.csv",
    JS / "13_curvature_by_sentence_layer.json",
    JS / "13_turning_angles.json",
    JS / "13_curvature_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

batch_mean.head()


## Next notebook

**17 — Cosine Alignment**

The next notebook should prototype the intervention:

\[
\mathcal{L}_{align}
=
1 -
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\]

Candidate outputs:

```text
17_alignment_loss_definition.png
17_lambda_sweep_plan.csv
17_alignment_training_stub.py
17_intervention_metrics.md
```

Core question:

```text
Can explicit mid-layer alignment amplify useful trajectory straightening
without degrading next-token prediction?
```
